# 03b2 — Seeded Multi-Run: DeiT-Base (×3)

deit_base trained 3x (seeds 42/123/2025), AdamW 1e-5, cosine, label smoothing 0.1. `num_workers=0` to avoid the dataloader hang. Each seed saves weights+preds immediately; re-run auto-skips completed seeds. Combine in 03c.

Attach: dataset + build_clean_split. Then Run All.

In [1]:
# ============================================================
# CONFIG
# ============================================================
DATA_ROOT = "/kaggle/input/datasets/shajinrp/diabetic-retinopathy/Dataset"
SPLIT_DIR = "/kaggle/input/notebooks/tochyokafor/build-clean-split"
OUT_DIR   = "/kaggle/working"
SEEDS     = [42, 123, 2025]
EPOCHS    = 10
BATCH     = 16
NUM_CLASSES = 5
NUM_WORKERS = 0

import os
for pth in [DATA_ROOT, SPLIT_DIR]:
    assert os.path.isdir(pth), f"Missing path: {pth}"
print("Paths OK | seeds:", SEEDS, "| workers:", NUM_WORKERS)

Paths OK | seeds: [42, 123, 2025] | workers: 0


In [2]:
!pip install timm --quiet
import os, random, numpy as np, torch, torch.nn as nn, torch.optim as optim, timm
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import (accuracy_score, f1_score,
    precision_recall_fscore_support, roc_auc_score)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_idx = np.load(f"{SPLIT_DIR}/clean_train_indices.npy").tolist()
val_idx   = np.load(f"{SPLIT_DIR}/clean_val_indices.npy").tolist()
test_idx  = np.load(f"{SPLIT_DIR}/clean_test_indices.npy").tolist()
class_names = open(f"{SPLIT_DIR}/class_names.txt").read().splitlines()
SEVERE_IDX = class_names.index("Severe")
print("Classes:", class_names, "| Severe idx:", SEVERE_IDX)

Device: cuda
Classes: ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe'] | Severe idx: 4


In [3]:
# --- full determinism per run ---
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    g = torch.Generator(); g.manual_seed(seed)
    return g

In [4]:
NORM = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
def make_transforms(size, crop_from=None):
    cf = crop_from or int(size * 1.15)
    train_tf = transforms.Compose([
        transforms.Resize((cf, cf)), transforms.RandomCrop(size),
        transforms.RandomHorizontalFlip(0.5), transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(20), transforms.ColorJitter(0.3,0.3,0.2,0.05),
        transforms.RandomAffine(0, translate=(0.05,0.05), scale=(0.95,1.05)),
        transforms.GaussianBlur(3, sigma=(0.1,1.0)),
        transforms.ToTensor(), NORM])
    test_tf = transforms.Compose([transforms.Resize((size,size)), transforms.ToTensor(), NORM])
    return train_tf, test_tf

def make_loaders(size, seed, crop_from=None):
    g = set_seed(seed)
    train_tf, test_tf = make_transforms(size, crop_from)
    train_ds = Subset(datasets.ImageFolder(DATA_ROOT, transform=train_tf), train_idx)
    test_ds  = Subset(datasets.ImageFolder(DATA_ROOT, transform=test_tf),  test_idx)
    return (DataLoader(train_ds, BATCH, shuffle=True, num_workers=NUM_WORKERS, generator=g),
            DataLoader(test_ds,  BATCH, shuffle=False, num_workers=NUM_WORKERS))

In [5]:
def train_model(model, loader, criterion, optimizer, scheduler=None, aux=False, epochs=EPOCHS):
    model.train()
    for ep in range(epochs):
        run = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            if aux and isinstance(out, tuple):
                loss = criterion(out[0], y) + 0.4 * criterion(out[1], y)
            else:
                loss = criterion(out.logits if hasattr(out,"logits") else out, y)
            loss.backward(); optimizer.step()
            run += loss.item()
        if scheduler: scheduler.step()
        print(f"    epoch {ep+1}/{epochs} loss {run/len(loader):.4f}")
    return model

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y = [], []
    for x, y in loader:
        out = model(x.to(device))
        out = out.logits if hasattr(out,"logits") else out
        P.extend(torch.softmax(out,1).cpu().numpy()); Y.extend(y.numpy())
    return np.array(P), np.array(Y)

def save_and_report(name, seed, model, probs, labels):
    preds = probs.argmax(1)
    acc = accuracy_score(labels, preds)
    f1m = f1_score(labels, preds, average="macro")
    pr,rc,f1,sup = precision_recall_fscore_support(labels,preds,labels=range(NUM_CLASSES),zero_division=0)
    torch.save(model.state_dict(), f"{OUT_DIR}/{name}_seed{seed}.pth")
    np.savez(f"{OUT_DIR}/{name}_seed{seed}_preds.npz", probs=probs, preds=preds, labels=labels)
    print(f"  [{name} seed {seed}] acc {acc:.4f} | macroF1 {f1m:.4f} | "
          f"SevereRec {rc[SEVERE_IDX]:.3f} | ProlifRec {rc[class_names.index('Proliferate_DR')]:.3f}")
    print(f"  saved {name}_seed{seed}.pth + _preds.npz")

def already_done(name, seed):
    p = f"{OUT_DIR}/{name}_seed{seed}_preds.npz"
    if os.path.exists(p):
        print(f"  [skip] {name} seed {seed} already done"); return True
    return False

### deit_base × 3

In [6]:
for s in SEEDS:
    if already_done("deit_base", s): continue
    print(f"deit_base seed {s}"); tr, te = make_loaders(224, s)
    m = timm.create_model("deit_base_patch16_224", pretrained=True, num_classes=NUM_CLASSES).to(device)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    opt = optim.AdamW(m.parameters(), lr=1e-5, weight_decay=0.05)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    m = train_model(m, tr, crit, opt, sch)
    probs, labels = evaluate(m, te); save_and_report("deit_base", s, m, probs, labels)
    del m; torch.cuda.empty_cache()

deit_base seed 42


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

    epoch 1/10 loss 0.6299
    epoch 2/10 loss 0.4465
    epoch 3/10 loss 0.4330
    epoch 4/10 loss 0.4282
    epoch 5/10 loss 0.4153
    epoch 6/10 loss 0.4106
    epoch 7/10 loss 0.4096
    epoch 8/10 loss 0.4079
    epoch 9/10 loss 0.4031
    epoch 10/10 loss 0.4033
  [deit_base seed 42] acc 0.9272 | macroF1 0.9165 | SevereRec 0.816 | ProlifRec 0.866
  saved deit_base_seed42.pth + _preds.npz
deit_base seed 123
    epoch 1/10 loss 0.6404
    epoch 2/10 loss 0.4468
    epoch 3/10 loss 0.4333
    epoch 4/10 loss 0.4258
    epoch 5/10 loss 0.4171
    epoch 6/10 loss 0.4147
    epoch 7/10 loss 0.4100
    epoch 8/10 loss 0.4055
    epoch 9/10 loss 0.4043
    epoch 10/10 loss 0.4056
  [deit_base seed 123] acc 0.9384 | macroF1 0.9257 | SevereRec 0.793 | ProlifRec 0.933
  saved deit_base_seed123.pth + _preds.npz
deit_base seed 2025
    epoch 1/10 loss 0.6264
    epoch 2/10 loss 0.4507
    epoch 3/10 loss 0.4326
    epoch 4/10 loss 0.4232
    epoch 5/10 loss 0.4216
    epoch 6/10 loss 0.4112

In [7]:
print("deit_base done. Save Version (Commit).")

deit_base done. Save Version (Commit).
